# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [2]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 31.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=c482a67b-edab-4067-855a-6032fb9c8f74
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 71.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 220MB/s]


In [3]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [4]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [5]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [6]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L-infinity PGD (use input_grad and clip_box)
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)

    for i in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    # TODO: implement L2 PGD (use input_grad, l2_norm and clip_box)
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp(min=1e-12)
        radius = torch.rand(x.size(0), 1, 1, 1, device=x.device)
        delta = delta * radius * eps
    else:
        delta = torch.zeros_like(x)
    delta = clip_box(x, delta)

    for i in range(steps):
        grad = input_grad(model, x, delta, y)
        grad = grad / l2_norm(grad).clamp(min=1e-12)
        delta = delta + alpha * grad
        norm = l2_norm(delta).clamp(min=1e-12)
        delta = delta * (eps / norm).clamp(max=1.0)
        delta = clip_box(x, delta)

    return (x + delta).detach()

In [7]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [8]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (151 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    # TODO: put the model in eval mode, attack with `norm`, put the model in train mode, and return the loss
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    loss = F.cross_entropy(model(x_adv), y)

    return loss

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (49 s)
epoch 2: loss 1.180 (98 s)
epoch 3: loss 1.151 (146 s)
epoch 4: loss 1.126 (195 s)
epoch 5: loss 1.097 (243 s)
epoch 6: loss 1.068 (292 s)
epoch 7: loss 1.049 (340 s)
epoch 8: loss 1.030 (389 s)
epoch 9: loss 1.013 (438 s)
epoch 10: loss 1.008 (486 s)
clean 78.7  linf 44.8  l2 57.0  l1 20.5  union 20.4   (148 s)


{'clean': 78.7, 'linf': 44.8, 'l2': 57.0, 'l1': 20.5, 'union': 20.4}

In [ ]:
original_eps = EPS["linf"]

for eps in [2 / 255, 4 / 255, 8 / 255, 16 / 255]:
    EPS["linf"] = eps
    print(f"\nTesting epsilon = {eps * 255:.0f}/255")
    test_model(model_linf)

EPS["linf"] = original_eps



Testing epsilon = 2/255
clean 78.7  linf 71.8  l2 57.0  l1 20.5  union 20.5   (148 s)

Testing epsilon = 4/255
clean 78.7  linf 62.9  l2 57.0  l1 20.5  union 20.5   (148 s)

Testing epsilon = 8/255
clean 78.7  linf 44.8  l2 57.0  l1 20.5  union 20.4   (148 s)

Testing epsilon = 16/255
clean 78.7  linf 15.4  l2 57.0  l1 20.5  union 12.7   (148 s)


In [ ]:
base_model = load_base_model()

print("Base Model:")
base_results = test_model(base_model)

print("Fine-tuned Model:")
trained_results = test_model(model_linf)

print(f"Base Clean: {base_results['clean']:.1f}%")
print(f"Trained Clean: {trained_results['clean']:.1f}%")
print(f"Base PGD: {base_results['linf']:.1f}%")
print(f"Trained PGD: {trained_results['linf']:.1f}%")

Base Model:
clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (148 s)
Fine-tuned Model:
clean 78.7  linf 44.8  l2 57.0  l1 20.5  union 20.4   (148 s)
Base Clean: 78.7%
Trained Clean: 78.7%
Base PGD: 48.4%
Trained PGD: 44.8%


In [ ]:
def test_fgsm(model, eps=8/255):
    model.eval()
    correct_clean = 0
    correct_fgsm = 0
    for i in range(0, len(x_test), 250):
        x = x_test[i:i + 250]
        y = y_test[i:i + 250]
        with torch.no_grad():
            correct_clean += (model(x).argmax(1) == y).sum().item()
        x_adv = pgd_linf(model, x, y, eps=eps, alpha=eps, steps=1, rand_init=False)

        with torch.no_grad():
            correct_fgsm += (model(x_adv).argmax(1) == y).sum().item()

    n = len(y_test)
    print(f"Clean Accuracy: {100 * correct_clean / n:.1f}%")
    print(f"FGSM Robust Accuracy: {100 * correct_fgsm / n:.1f}%")


print("Base Model:")
test_fgsm(base_model)

print("Fine-tuned Model:")
test_fgsm(model_linf)


Base Model:
Clean Accuracy: 78.7%
FGSM Robust Accuracy: 52.5%
Fine-tuned Model:
Clean Accuracy: 78.7%
FGSM Robust Accuracy: 49.4%


In [ ]:
original_eps = EPS["linf"]

for eps in [2/255, 4/255, 8/255, 16/255]:
    EPS["linf"] = eps
    print(f"\nBase Model, epsilon = {eps * 255:.0f}/255")
    results = test_model(base_model)
    print(f"PGD Robust Accuracy: {results['linf']:.1f}%")

EPS["linf"] = original_eps



Base Model, epsilon = 2/255
clean 78.7  linf 72.1  l2 59.8  l1 22.4  union 22.4   (148 s)
PGD Robust Accuracy: 72.1%

Base Model, epsilon = 4/255
clean 78.7  linf 64.7  l2 59.8  l1 22.4  union 22.4   (148 s)
PGD Robust Accuracy: 64.7%

Base Model, epsilon = 8/255
clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (148 s)
PGD Robust Accuracy: 48.4%

Base Model, epsilon = 16/255
clean 78.7  linf 17.4  l2 59.8  l1 22.4  union 14.4   (148 s)
PGD Robust Accuracy: 17.4%


In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (49 s)
epoch 2: loss 0.825 (97 s)
epoch 3: loss 0.787 (146 s)
epoch 4: loss 0.754 (196 s)
epoch 5: loss 0.714 (245 s)
epoch 6: loss 0.675 (294 s)
epoch 7: loss 0.631 (343 s)
epoch 8: loss 0.610 (391 s)
epoch 9: loss 0.591 (439 s)
epoch 10: loss 0.583 (488 s)
clean 82.4  linf 32.1  l2 59.9  l1 28.0  union 24.4   (148 s)


{'clean': 82.4, 'linf': 32.1, 'l2': 59.9, 'l1': 28.0, 'union': 24.4}

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.235 (51 s)
epoch 2: loss 0.849 (98 s)
epoch 3: loss 0.581 (148 s)
epoch 4: loss 0.454 (196 s)
epoch 5: loss 0.349 (245 s)
epoch 6: loss 0.270 (294 s)
epoch 7: loss 0.220 (343 s)
epoch 8: loss 0.177 (392 s)
epoch 9: loss 0.165 (441 s)
epoch 10: loss 0.156 (490 s)
clean 84.2  linf 0.0  l2 0.0  l1 29.7  union 0.0   (148 s)


{'clean': 84.2, 'linf': 0.0, 'l2': 0.0, 'l1': 29.7, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [16]:
def ramp_loss(model, x, y, lam=0.5):
    # TODO: in eval mode, attack with linf and l1; in train mode, return the max of the two losses plus lam * KL
    model.eval()

    x_linf = attack(model, x, y, "linf")
    x_l1 = attack(model, x, y, "l1")

    model.train()

    logits_linf = model(x_linf)
    logits_l1 = model(x_l1)
    adv_loss = torch.maximum(F.cross_entropy(logits_linf, y, reduction="none"), F.cross_entropy(logits_l1, y, reduction="none")).mean()
    correct_mask = logits_linf.argmax(dim=1) == y

    if correct_mask.any():
        kl_loss = F.kl_div(F.log_softmax(logits_linf[correct_mask], dim=1), F.softmax(logits_l1[correct_mask], dim=1), reduction="batchmean")
    else:
        kl_loss = logits_linf.sum() * 0.0

    adv_loss += lam * kl_loss

    return adv_loss


def gradient_projection(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    cosine = F.cosine_similarity(g_n.reshape(-1).unsqueeze(0), g_a.reshape(-1).unsqueeze(0), dim=1, eps=1e-12).squeeze(0)

    return torch.clamp(cosine, min=0.0) * g_n

In [ ]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.435 (105 s)
epoch 2: loss 1.331 (209 s)
epoch 3: loss 1.304 (313 s)
epoch 4: loss 1.281 (417 s)
epoch 5: loss 1.254 (522 s)
epoch 6: loss 1.227 (625 s)
epoch 7: loss 1.199 (729 s)
epoch 8: loss 1.186 (833 s)
epoch 9: loss 1.172 (936 s)
epoch 10: loss 1.166 (1040 s)
clean 79.2  linf 40.7  l2 61.8  l1 44.6  union 38.9   (148 s)


{'clean': 79.2, 'linf': 40.7, 'l2': 61.8, 'l1': 44.6, 'union': 38.9}

# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [10]:
def my_loss(model, x, y):
    # TODO: your method
    step = ans_state["step"]

    if step < len(ANS_NORMS):
        norm_idx = step
    else:
        robustness = torch.tensor([ans_state["ema"][n] for n in ANS_NORMS], dtype=torch.float32)
        weakness = (1.0 - robustness).pow(2)
        prob = weakness / weakness.sum()
        prob = (1.0 - ans_state["exploration"]) * prob + ans_state["exploration"] / len(ANS_NORMS)
        norm_idx = torch.multinomial(prob, num_samples=1).item()

    norm = ANS_NORMS[norm_idx]
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()

    logits = model(x_adv)

    with torch.no_grad():
        batch_accuracy = (logits.argmax(dim=1) == y).float().mean().item()

    momentum = ans_state["momentum"]

    if ans_state["counts"][norm] == 0:
        ans_state["ema"][norm] = batch_accuracy
    else:
        ans_state["ema"][norm] = momentum * ans_state["ema"][norm] + (1.0 - momentum) * batch_accuracy

    ans_state["counts"][norm] += 1
    ans_state["step"] += 1

    return F.cross_entropy(logits, y)

In [13]:
ANS_NORMS = ("linf", "l2", "l1")

ans_state = {
    "ema": {norm: 0.5 for norm in ANS_NORMS},
    "counts": {norm: 0 for norm in ANS_NORMS},
    "step": 0,
    "momentum": 0.9,
    "exploration": 0.15
}

model_ans_0 = adv_train(load_base_model(), my_loss, seed=0)

print("Sampling counts:")
print(ans_state["counts"])

print("Final EMA robustness:")
print(ans_state["ema"])

robustness = torch.tensor([ans_state["ema"][n] for n in ANS_NORMS], dtype=torch.float32)
weakness = (1.0 - robustness).pow(2)
prob = weakness / weakness.sum()
prob = (1.0 - ans_state["exploration"]) * prob + ans_state["exploration"] / len(ANS_NORMS)

print("Final sampling probabilities:")
print({norm: round(p, 4) for norm, p in zip(ANS_NORMS, prob.tolist())})

print("Evaluation:")
ans_results_0 = test_model(model_ans_0)

epoch 1: loss 1.262 (40 s)
epoch 2: loss 1.180 (82 s)
epoch 3: loss 1.135 (127 s)
epoch 4: loss 1.098 (170 s)
epoch 5: loss 1.060 (214 s)
epoch 6: loss 1.061 (258 s)
epoch 7: loss 1.032 (302 s)
epoch 8: loss 1.001 (346 s)
epoch 9: loss 0.990 (389 s)
epoch 10: loss 0.991 (433 s)
Sampling counts:
{'linf': 356, 'l2': 149, 'l1': 275}
Final EMA robustness:
{'linf': 0.5796709972867121, 'l2': 0.7432832511530618, 'l1': 0.629400406309873}
Final sampling probabilities:
{'linf': 0.4453, 'l2': 0.1974, 'l1': 0.3573}
Evaluation:
clean 78.1  linf 41.7  l2 62.2  l1 47.5  union 40.5   (134 s)


In [14]:
ans_state = {
    "ema": {norm: 0.5 for norm in ANS_NORMS},
    "counts": {norm: 0 for norm in ANS_NORMS},
    "step": 0,
    "momentum": 0.9,
    "exploration": 0.15
}

model_ans_1 = adv_train(load_base_model(), my_loss, seed=1)

print("Sampling counts:")
print(ans_state["counts"])

print("Final EMA robustness:")
print(ans_state["ema"])

robustness = torch.tensor([ans_state["ema"][n] for n in ANS_NORMS], dtype=torch.float32)
weakness = (1.0 - robustness).pow(2)
prob = weakness / weakness.sum()
prob = (1.0 - ans_state["exploration"]) * prob + ans_state["exploration"] / len(ANS_NORMS)

print("Final sampling probabilities:")
print({norm: round(p, 4) for norm, p in zip(ANS_NORMS, prob.tolist())})

print("Evaluation:")
ans_results_1 = test_model(model_ans_1)

epoch 1: loss 1.231 (44 s)
epoch 2: loss 1.176 (88 s)
epoch 3: loss 1.123 (132 s)
epoch 4: loss 1.102 (176 s)
epoch 5: loss 1.093 (220 s)
epoch 6: loss 1.054 (264 s)
epoch 7: loss 1.035 (308 s)
epoch 8: loss 0.985 (352 s)
epoch 9: loss 1.014 (396 s)
epoch 10: loss 0.970 (440 s)
Sampling counts:
{'linf': 344, 'l2': 164, 'l1': 272}
Final EMA robustness:
{'linf': 0.5566043942370686, 'l2': 0.7462908432164331, 'l1': 0.6218785456337487}
Final sampling probabilities:
{'linf': 0.4637, 'l2': 0.1854, 'l1': 0.3509}
Evaluation:
clean 77.9  linf 41.2  l2 62.2  l1 47.3  union 39.4   (133 s)


In [17]:
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp_1)

epoch 1: loss 1.436 (94 s)
epoch 2: loss 1.333 (188 s)
epoch 3: loss 1.299 (281 s)
epoch 4: loss 1.274 (374 s)
epoch 5: loss 1.248 (468 s)
epoch 6: loss 1.222 (561 s)
epoch 7: loss 1.200 (654 s)
epoch 8: loss 1.186 (747 s)
epoch 9: loss 1.178 (841 s)
epoch 10: loss 1.169 (934 s)
clean 78.9  linf 42.3  l2 62.7  l1 46.7  union 40.8   (134 s)


{'clean': 78.9, 'linf': 42.3, 'l2': 62.7, 'l1': 46.7, 'union': 40.8}

In [19]:
def ablation_loss(model, x, y):
    norm = ANS_NORMS[torch.randint(0, len(ANS_NORMS), (1,)).item()]
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()

    return F.cross_entropy(model(x_adv), y)

In [20]:
model_ablation_0 = adv_train(load_base_model(), ablation_loss, seed=0)
uniform_results_0 = test_model(model_ablation_0)

epoch 1: loss 1.167 (45 s)
epoch 2: loss 1.120 (88 s)
epoch 3: loss 1.055 (132 s)
epoch 4: loss 1.028 (176 s)
epoch 5: loss 1.027 (220 s)
epoch 6: loss 0.975 (264 s)
epoch 7: loss 0.970 (308 s)
epoch 8: loss 0.977 (352 s)
epoch 9: loss 0.926 (396 s)
epoch 10: loss 0.899 (439 s)
clean 78.5  linf 40.5  l2 62.4  l1 46.6  union 39.0   (134 s)


In [21]:
model_ablation_1 = adv_train(load_base_model(), ablation_loss, seed=1)
uniform_results_1 = test_model(model_ablation_1)

epoch 1: loss 1.204 (45 s)
epoch 2: loss 1.111 (88 s)
epoch 3: loss 1.046 (132 s)
epoch 4: loss 1.042 (176 s)
epoch 5: loss 1.004 (220 s)
epoch 6: loss 1.003 (264 s)
epoch 7: loss 0.960 (308 s)
epoch 8: loss 0.944 (351 s)
epoch 9: loss 0.948 (395 s)
epoch 10: loss 0.967 (439 s)
clean 79.1  linf 40.9  l2 62.5  l1 46.5  union 39.1   (133 s)


In [22]:
ans_strong_0 = test_model(model_ans_0, steps=200, restarts=5)
ans_strong_1 = test_model(model_ans_1, steps=200, restarts=5)

clean 78.1  linf 41.7  l2 62.1  l1 45.6  union 40.0   (1325 s)
clean 77.9  linf 41.0  l2 62.1  l1 46.2  union 39.2   (1325 s)


In [ ]:
# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)